# Scala en Binder: prueba mínima

**Objetivo:** comprobar el kernel Scala, el estado entre celdas y las operaciones que utilizaremos en el tutorial.

Este notebook combina la prueba técnica del entorno con el bloque de colecciones del tutorial. Ejecuta de arriba hacia abajo con **Shift + Enter**, o usa **Kernel → Restart Kernel and Run All Cells**. Selecciona **Scala 3.9**.

Caso: un catálogo de productos. Calcularemos el precio de comprar **una unidad de cada producto disponible**, con un descuento común. Importes en pesos colombianos (COP).

**Resultado esperado con 10 %:** 315000 COP y `PRUEBA COMPLETA: OK` al final.

Binder es temporal: descarga el notebook si quieres conservar tus cambios. Guardarlo aquí no actualiza GitHub.

## 1. Primera ejecución — integrante 1

Estas dos celdas prueban que una definición de Scala permanece disponible para la siguiente. La función usa sintaxis con indentación de Scala 3.

In [ ]:
val mensaje = "Scala funciona"
val numeroDePrueba = 21

def duplicar(n: Int): Int =
  n * 2

println(mensaje)

In [ ]:
assert(duplicar(numeroDePrueba) == 42)
println("ESTADO ENTRE CELDAS: OK")

## 2. Función sobre un precio — integrante 2

Contrato provisional: precio válido no negativo y porcentaje entero entre 0 y 100. `BigDecimal` se construye con texto para representar los decimales del ejemplo. La política de redondeo comercial queda fuera de esta prueba.

`porcentajeDescuento` será utilizado por los bloques posteriores.

In [ ]:
val porcentajeDescuento = 10

def aplicarDescuento(precio: BigDecimal, porcentaje: Int): BigDecimal =
  precio * (BigDecimal(100) - BigDecimal(porcentaje)) / BigDecimal(100)

assert(porcentajeDescuento >= 0 && porcentajeDescuento <= 100)
assert(aplicarDescuento(BigDecimal("100000"), 10) == BigDecimal("90000"))
assert(aplicarDescuento(BigDecimal("100000"), 0) == BigDecimal("100000"))
assert(aplicarDescuento(BigDecimal("100000"), 100) == BigDecimal(0))
println("FUNCION DE DESCUENTO: OK")

## 3. Modelo y catálogo inmutable — integrante 3

Campos acordados como punto de partida: identificador, nombre, precio y disponibilidad. Todos los precios son válidos; el programa no procesa textos mal formados.

El mouse no está disponible y no se incluirá en el total.

In [ ]:
case class Producto(id: Int, nombre: String, precio: BigDecimal, disponible: Boolean)

val productos = List(
  Producto(1, "Teclado", BigDecimal("100000"), true),
  Producto(2, "Audifonos", BigDecimal("200000"), true),
  Producto(3, "Mouse", BigDecimal("80000"), false),
  Producto(4, "Soporte", BigDecimal("50000"), true)
)

assert(productos.size == 4)
productos.foreach(println)

In [ ]:
val tecladoOriginal = productos.head
val tecladoEnOferta = tecladoOriginal.copy(
  precio = aplicarDescuento(tecladoOriginal.precio, porcentajeDescuento)
)

assert(tecladoOriginal.precio == BigDecimal("100000"))
assert(tecladoEnOferta.precio == aplicarDescuento(BigDecimal("100000"), porcentajeDescuento))
println(s"Original: ${tecladoOriginal.precio}; copia: ${tecladoEnOferta.precio}")

## 4. ¿Qué incluye la cotización?

La tienda ofrece un **10 % de descuento**. Queremos cotizar **una unidad de cada producto disponible**, mostrar el nombre y el precio final de cada uno y calcular el total. Reutilizaremos `productos` del bloque 3 y `aplicarDescuento` del bloque 2.

| Producto | Precio original (COP) | Disponible |
|---|---:|:---:|
| Teclado | 100.000 | Sí |
| Audífonos | 200.000 | Sí |
| Mouse | 80.000 | No |
| Soporte | 50.000 | Sí |

Primero resolveremos la cotización con un bucle. Después separaremos sus tres decisiones en **seleccionar → transformar → acumular** y comprobaremos que el detalle y el total coinciden. Antes de ejecutar, predice qué productos se incluirán.

### 4.1. Resolver la cotización con un bucle imperativo

El `for` visita cada producto y el `if` decide si se incluye. Para los disponibles, el bucle calcula el precio final, añade un par `(nombre, precioFinal)` y actualiza la suma. Antes de ejecutar, predice qué productos aparecerán y calcula el total.

`totalImperativo` es un `var` porque se reasigna. `lineasImperativas` es un `val` que apunta a un `ListBuffer` cuyo contenido sí cambia. Ambos acumuladores son locales a la llamada. Al finalizar, el método devuelve el detalle como lista inmutable y el total.

Se combinan tres tareas dentro del bucle. Si agregamos un tope por precio final, la condición iría después de calcular el descuento y antes de actualizar tanto el detalle como el total. Ahora vamos a expresar cada tarea por separado.

In [ ]:
def cotizarImperativo(catalogo: List[Producto], descuento: Int)
    : (List[(String, BigDecimal)], BigDecimal) =
  var totalImperativo = BigDecimal(0)
  val lineasImperativas = scala.collection.mutable.ListBuffer.empty[(String, BigDecimal)]

  for (producto <- catalogo) {
    if (producto.disponible) {
      val precioFinal = aplicarDescuento(producto.precio, descuento)
      lineasImperativas += ((producto.nombre, precioFinal))
      totalImperativo += precioFinal
    }
  }

  (lineasImperativas.toList, totalImperativo)

val (detalleImperativo, totalImperativo) =
  cotizarImperativo(productos, porcentajeDescuento)

detalleImperativo.foreach(linea => println(s"${linea._1}: ${linea._2} COP"))
println(s"Total imperativo: $totalImperativo COP")

### 4.2. Seleccionar con `filter`

La condición de disponibilidad del bucle pasa a ser una función que podemos usar por separado.

`filter` recibe una función que contesta sí o no para cada producto. En `producto => producto.disponible`, la flecha separa el parámetro del cuerpo y el resultado es un `Boolean`. Si la función devuelve `true`, se conserva el producto completo. Por eso pasamos de `List[Producto]` a otra `List[Producto]`, con los IDs 1, 2 y 4.

Una operación que recibe una función como argumento es de **orden superior**. En este caso podemos escribir `_.disponible` como abreviatura de la lambda. Usaremos el nombre explícito durante la demostración.

In [ ]:
val disponibles = productos.filter(producto => producto.disponible)
assert(disponibles.map(producto => producto.id) == List(1, 2, 4))
println(s"Disponibles: ${disponibles.map(producto => producto.nombre)}")

### 4.3. Transformar con `map`

`map` produce una línea de cotización por cada producto seleccionado. Cada línea es la tupla `(nombre, precioFinal)`. El alias `LineaCotizacion` permite nombrar ese par en las firmas. Sigue siendo un `(String, BigDecimal)` y sus componentes se leen con `._1` y `._2`. El precio del catálogo se conserva.

`crearLineas` utiliza dos listas de parámetros. La primera fija el descuento y la segunda recibe los productos. Esta forma, conocida como **currificación** o *currying*, permite una **aplicación parcial**. Al escribir `crearLineas(10)`, obtenemos una función `List[Producto] => List[LineaCotizacion]` lista para usar en distintos catálogos. El `map` se ejecuta cuando le pasamos una lista de productos.

**Antes de ejecutar:** ¿qué dato perderíamos si el `map` devolviera solo el precio final?

In [ ]:
type LineaCotizacion = (String, BigDecimal)

def crearLinea(descuento: Int)(producto: Producto): LineaCotizacion =
  (producto.nombre, aplicarDescuento(producto.precio, descuento))

def crearLineas(descuento: Int)(seleccion: List[Producto]): List[LineaCotizacion] =
  seleccion.map(crearLinea(descuento))

val cotizarCampana: List[Producto] => List[LineaCotizacion] =
  crearLineas(porcentajeDescuento)

val lineas = cotizarCampana(disponibles)
lineas.foreach(linea => println(s"${linea._1}: ${linea._2} COP"))

### 4.4. Acumular con `foldLeft`

`foldLeft` empieza con un valor inicial y recorre la lista de izquierda a derecha. En cada paso, la función recibe el **acumulado** y la **línea actual**. Devuelve el nuevo acumulado al sumar el segundo componente de la línea. El valor inicial es `BigDecimal(0)` porque la suma empieza en cero y los precios son `BigDecimal`.

| Paso con 10 % | Acumulado anterior | Precio final | Nuevo acumulado |
|---|---:|---:|---:|
| Teclado | 0 | 90.000 | 90.000 |
| Audífonos | 90.000 | 180.000 | 270.000 |
| Soporte | 270.000 | 45.000 | 315.000 |

Si la lista está vacía, la función nunca se llama y se devuelve el valor inicial. Para sumar una lista de precios también se puede usar `sum`. Aquí `foldLeft` deja visible cómo se construye el resultado.

In [ ]:
def sumarLineas(detalle: List[LineaCotizacion]): BigDecimal =
  detalle.foldLeft(BigDecimal(0)) { (acumulado, linea) =>
    acumulado + linea._2
  }

val total = sumarLineas(lineas)
val factorEsperado = (BigDecimal(100) - BigDecimal(porcentajeDescuento)) / BigDecimal(100)
assert(total == BigDecimal("350000") * factorEsperado)
assert(productos.head.precio == BigDecimal("100000"))
println(s"Total con $porcentajeDescuento % de descuento: $total COP")

### Comprobar que las dos soluciones coinciden

**Predice antes de ejecutar:** ¿basta con que los totales sean iguales? Comparamos la lista completa para comprobar también los nombres, los precios y el orden. El caso vacío debe producir un detalle vacío y un total de cero.

En la cadena funcional, cada operación expresa una decisión y deja un resultado que podemos reutilizar o probar. En el reto decidirás en qué punto conviene comprobar el tope. En el bucle, esa regla debe proteger las dos actualizaciones.

El bucle es secuencial y su mutación es local. Si varios hilos actualizan el mismo búfer o total sin coordinación, pueden aparecer condiciones de carrera. Las transformaciones puras evitan esa mutación compartida. Para paralelizar se necesita además una estrategia de ejecución y una forma válida de combinar resultados parciales.

In [ ]:
assert(detalleImperativo == lineas)
assert(totalImperativo == total)
assert(cotizarImperativo(List.empty[Producto], porcentajeDescuento) == (Nil, BigDecimal(0)))
println(s"Bucle y cadena funcional coinciden en detalle y total ($totalImperativo COP).")

### 4.5. Evaluación estricta y perezosa con `view`

Con una `List` estricta, `filter` recorre el catálogo y entrega una lista de disponibles. Después `map` recorre esa selección y construye la lista de líneas. Finalmente `foldLeft` consume las líneas y devuelve un solo total. Son tres etapas de recorrido sobre colecciones de tamaños distintos. Las etapas que entregan listas son el filtro y el mapa.

Con `view`, `filter` y `map` se difieren. Cuando `foldLeft` pide elementos, la vista prueba la disponibilidad y transforma cada producto aceptado al consumirlo. Para calcular **solo el total**, este encadenamiento recorre la entrada una vez y evita materializar las listas intermedias. Sigue siendo secuencial y una mejora de tiempo se tendría que medir.

Para mostrar las líneas y reutilizarlas, la lista estricta sigue siendo útil. Consumir una vista dos veces puede repetir el trabajo. Por eso conviene que las funciones de transformación sean puras, y dejar la impresión para después del cálculo.

In [ ]:
def totalDesdeVista(catalogo: List[Producto], descuento: Int): BigDecimal =
  catalogo.view
    .filter(producto => producto.disponible)
    .map(crearLinea(descuento))
    .foldLeft(BigDecimal(0)) { (acumulado, linea) =>
      acumulado + linea._2
    }

val totalConVista = totalDesdeVista(productos, porcentajeDescuento)
assert(totalConVista == total)
assert(totalDesdeVista(List.empty[Producto], porcentajeDescuento) == BigDecimal(0))
println(s"Lista estricta y vista producen el mismo total ($totalConVista COP).")

In [ ]:
val catalogoVacio = List.empty[Producto]
val totalVacio = sumarLineas(cotizarCampana(catalogoVacio))
assert(totalVacio == BigDecimal(0))
println("CATALOGO VACIO: OK (total 0)")

### Reto en vivo · Un tope por precio final (2 min)

El cliente quiere cotizar únicamente productos **disponibles** cuyo **precio por unidad después del descuento** sea menor o igual al tope. El reto usa un 10 % fijo para que todos trabajen con los mismos resultados.

1. Predice si el teclado entra con un tope de 90.000 COP. Su precio original es 100.000 COP.
2. En la siguiente celda, completa el cuerpo de `seleccionarConTope` usando `filter`. Usa el precio de cada línea y el parámetro `tope`. Ejecuta esa celda y la de comprobación.
3. Explica por qué un filtro sobre el precio original excluiría un producto que sí debería entrar. Explica también por qué el mouse sigue fuera.

| Tope por unidad | Productos esperados | Total esperado |
|---:|---|---:|
| 90.000 COP | Teclado y Soporte | 135.000 COP |
| 89.999 COP | Soporte | 45.000 COP |
| 40.000 COP | Ninguno | 0 COP |

**Pista:** las líneas que recibe la función ya vienen de seleccionar disponibles y calcular el descuento. Un tope por unidad puede producir un total mayor que el tope.

In [ ]:
def seleccionarConTope(detalle: List[LineaCotizacion], tope: BigDecimal)
    : List[LineaCotizacion] =
  detalle // TODO: devuelve solo las lineas cuyo precio final cumple el tope.

In [ ]:
val lineasReto = crearLineas(10)(disponibles)
val casosTope = List(
  (BigDecimal("90000"), List("Teclado", "Soporte"), BigDecimal("135000")),
  (BigDecimal("89999"), List("Soporte"), BigDecimal("45000")),
  (BigDecimal("40000"), List.empty[String], BigDecimal(0))
)

casosTope.foreach { caso =>
  val (tope, nombresEsperados, totalEsperado) = caso
  val seleccion = seleccionarConTope(lineasReto, tope)
  val nombres = seleccion.map(linea => linea._1)
  val importe = sumarLineas(seleccion)
  val pasa = nombres == nombresEsperados && importe == totalEsperado
  val estado = if pasa then "OK" else "Por revisar"
  println(s"$estado | Tope: $tope COP | Productos: $nombres | Total: $importe COP")
}

<details>
<summary>Solución del reto en vivo</summary>

En el cuerpo de `seleccionarConTope`:

```scala
detalle.filter(linea => linea._2 <= tope)
```

La tupla ya contiene el precio final. El teclado entra a 90.000 COP por el `<=` y queda fuera a 89.999 COP. El mouse quedó fuera al filtrar la disponibilidad, antes de construir las líneas. Con 40.000 COP queda la lista vacía y la suma devuelve cero.

</details>

### Reto avanzado · Reporte de precios por categoría

La tienda quiere saber **cuántos productos disponibles** hay por categoría y **cuál es su precio final promedio** con un descuento del 10 %. Asocia cada ID con una categoría sin cambiar `Producto`. Incluye Periféricos (IDs 1, 2 y 3) y Accesorios (ID 4). El mouse agotado pertenece a Periféricos.

| Categoría | Cantidad disponible | Precio final promedio |
|---|---:|---:|
| Periféricos | 2 | 135.000 COP |
| Accesorios | 1 | 45.000 COP |

**Pistas:** sobre `disponibles`, usa `groupBy` pasando una función que devuelva la categoría de cada producto. El resultado es un `Map[String, List[Producto]]`. Transforma cada grupo en líneas de cotización, suma los precios y divide por el número de líneas. Al agrupar los disponibles, cada grupo existente tiene al menos un producto.

<details>
<summary>Solución del reporte por categoría</summary>

```scala
val categoriaPorId = Map(
  1 -> "Periféricos", 2 -> "Periféricos",
  3 -> "Periféricos", 4 -> "Accesorios"
)

val grupos = disponibles.groupBy(producto => categoriaPorId(producto.id))
val reporte = grupos.map { case (categoria, grupo) =>
  val detalle = crearLineas(10)(grupo)
  val cantidad = detalle.size
  val promedio = sumarLineas(detalle) / BigDecimal(cantidad)
  categoria -> (cantidad, promedio)
}

assert(reporte == Map(
  "Periféricos" -> (2, BigDecimal("135000")),
  "Accesorios" -> (1, BigDecimal("45000"))
))
reporte.toList.sortBy(entrada => entrada._1).foreach(println)
```

El mouse se excluye antes de formar los grupos. Cada clave producida por `groupBy` tiene al menos un producto, y `crearLineas` produce exactamente una línea por producto. Por eso `cantidad > 0` y el promedio se puede calcular directamente. Si la entrada es vacía, el mapa de grupos queda vacío y no se ejecuta la división. El tipo devuelto sigue siendo `Map[String, List[Producto]]`. Que los grupos sean no vacíos es una propiedad de cómo se construyen. El compilador los sigue tratando como listas ordinarias.

</details>

### Dos ampliaciones para el tutorial

**Catálogos de dos sucursales.** Construye una segunda lista con un teclado del mismo ID y un precio diferente, y un producto nuevo con ID 5. Une ambas listas y, si hay IDs repetidos, conserva el primer registro. Comprueba que el teclado conserva el precio de la primera sucursal y que aparece el producto nuevo. La combinación `++` pone los elementos de la segunda lista al final. En esta lista, `distinctBy` conserva la primera aparición de cada clave.

<details>
<summary>Solución para combinar catálogos</summary>

```scala
val sucursalDos = List(
  productos.head.copy(precio = BigDecimal("120000")),
  Producto(5, "Cámara", BigDecimal("150000"), true)
)

val catalogoUnido = (productos ++ sucursalDos).distinctBy(producto => producto.id)
assert(catalogoUnido.map(producto => producto.id) == List(1, 2, 3, 4, 5))
assert(catalogoUnido.head.precio == BigDecimal("100000"))
```

</details>

**Seleccionar y transformar con `collect`.** Después de leer el bloque 5, intenta obtener las mismas líneas de la cotización en una sola operación. Una función parcial expresa qué elementos acepta y cómo transforma los aceptados. Si la condición del `case` falla, `collect` omite ese elemento.

<details>
<summary>Solución con collect</summary>

```scala
val lineasConCollect = productos.collect {
  case producto if producto.disponible =>
    (producto.nombre, aplicarDescuento(producto.precio, porcentajeDescuento))
}

assert(lineasConCollect == lineas)
```

</details>

**Puente al bloque 5:** la cotización solo lista productos disponibles. Una búsqueda por ID debe consultar el catálogo original para poder encontrar también un producto agotado.

## 5. Búsqueda con presencia o ausencia — integrante 5

La búsqueda devuelve `Option[Producto]`. Consultar por identificador es independiente de que el producto esté disponible.

In [ ]:
def buscarProducto(id: Int): Option[Producto] =
  productos.find(producto => producto.id == id)

def describirResultado(resultado: Option[Producto]): String =
  resultado match
    case Some(producto) => s"Encontrado: ${producto.nombre}"
    case None => "Producto no encontrado"

assert(buscarProducto(1).contains(tecladoOriginal))
assert(buscarProducto(999).isEmpty)
assert(describirResultado(buscarProducto(999)) == "Producto no encontrado")
println(describirResultado(buscarProducto(1)))
println(describirResultado(buscarProducto(999)))

In [ ]:
println("PRUEBA COMPLETA: OK")

## Comprobación manual y siguiente paso

1. Cambiar `porcentajeDescuento` de 10 a 20 y vuelve a ejecutar desde la celda de esa definición hasta el final. El total debe ser **280000 COP**.
2. Restaurar 10, reinicia el kernel y ejecuta todo en orden. Debe volver a dar **315000 COP**.
3. Descargar el notebook para conservar tu trabajo.
